# Paper 2, Session 8: second test window (robustness check)

This notebook repeats the short-history part of the main experiment on a **second, earlier test window**:
26 weeks from **16 June to 14 December 2025** (window 1 was 15 December 2025 to 14 June 2026). It answers the reviewer question
"does the time scale of the transfer benefit hold in another season?".

* Everything is moved 26 weeks earlier: data after the new window are dropped, target histories end at the new test start,
  source pretraining uses only weeks before it.
* Budgets 0 (zero-shot), 4, 8 and 16 weeks; 3 seeds; models Prior, Hist, LogReg, GBDT, GBDT-pool, TabM-mini, FTT, FTT-pre, TabICL,
  HiST-MAT-X (scratch, pretrained, zero-shot). Code, features and hyperparameters are identical to the main experiment (protocol v3).
* Only the raw data files of the main experiment are reused (no re-download). Its results and pretrained models are NOT reused.

**Run:** Accelerator *GPU T4 x2*, Internet *On*. Input: the main experiment notebook output (Session 6). Save Version → Save & Run All.
It needs about 10 GPU hours. It stops starting new runs after `TIME_LIMIT_HOURS` and saves everything. If it did not finish,
run it again (Session 8b) with **both** inputs: the Session 6 output and this notebook's own previous output; finished runs are skipped.
At the end download `histmatx_w2.zip` from the Output panel.

## 1. Configuration

In [ ]:
import os, sys, json, time, math, gc, re, copy, zipfile, shutil, warnings, hashlib, subprocess
from pathlib import Path
import numpy as np, pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)

RUN_MODE   = os.environ.get("HMX_MODE", "full")   # "full" | "fast" | "synthetic"
# Session 8 (window 2): one session for all three cities; if time runs out, run again with this output added as input (Session 8b).
TARGETS    = os.environ.get("HMX_TARGETS", "SF,CHI,NYC").split(",")   # all three; a second session resumes where the first stopped
RUN_BUDGETS = None                       # None = all budgets (finished runs are skipped)
RUN_MAIN = True
WINDOW_SHIFT = 26     # SECOND TEST WINDOW: the whole time grid is moved 26 weeks earlier (test 16 Jun - 14 Dec 2025)
# Finds the previous session's output automatically, wherever it is under /kaggle/input (notebook output, Dataset or Model).
# Data + pretrained models and results may sit in different places (a Dataset made from notebook output can drop the
# 'tables'/'runs' folders but keeps histmatx_outputs.zip), so the pieces are collected separately into one resume folder.
import glob, zipfile as _zf
INPUT_ROOT = os.environ.get("HMX_INPUT_ROOT", "/kaggle/input")
def _collect_resume(root):
    """Window 2 reuses ONLY the raw data of the main experiment (data/*.csv). Results, runs and pretrained models of window 1
    are never reused, because they were trained with the other test window. A previous window-2 session (folder 'histmatx_w2')
    is resumed completely."""
    stage = Path("/tmp/hmx_resume"); shutil.rmtree(stage, ignore_errors=True); stage.mkdir(parents=True)
    def best(pattern, score):
        c = glob.glob(f"{root}/**/{pattern}", recursive=True)
        return max(c, key=score) if c else None
    dat = best("data/CHI.csv", lambda p: len(list(Path(p).parent.glob("*.csv"))))
    w2 = best("histmatx_w2/tables/results_main.csv", lambda p: sum(1 for _ in open(p)))
    w2root = Path(w2).parents[1] if w2 else None
    for sub, s in [("data", Path(dat).parent if dat else None)] + [(x, w2root / x if w2root else None) for x in ["tables", "runs", "logs", "models"]]:
        if s is not None and s.exists(): (stage / sub).symlink_to(s.resolve(), target_is_directory=True)
        print(f"  resume {sub:7s} <- {s}")
    return str(stage) if dat else ""
RESUME_FROM = _collect_resume(INPUT_ROOT) if Path(INPUT_ROOT).exists() else ""
print("RESUME_FROM =", RESUME_FROM or "NOT FOUND (fine only for the very first session)")
# Runs to recompute (target, budget, models, recompute if protocol < this). Leave as is.
#  v2: epoch floor (Session 1 hit the epoch cap at budgets 52/all)
#  v3: zero-shot masks premises a city does not record (SF has none) -- the design said so, the code did not
_DEEP = ["MLP", "TabM-mini", "FTT", "FTT-pre", "HMX-scratch", "HMX-pre"]
RERUN_RULES = []
PROTOCOL_VERSION = 3
EXPECTED_TEST_WINDOW = (337, 362) if RUN_MODE == "full" else None   # window 2: 2025-06-16 to 2025-12-14 (window 1 was 363-388)
TIME_LIMIT_HOURS = float(os.environ.get("HMX_TIME_LIMIT", 10.5))   # lower this if your remaining GPU quota is < 11.5 h                           # stop starting new runs after this; analysis + export still run (Kaggle kills at 12 h)
_T_START = time.time()
def time_up(): return (time.time() - _T_START) / 3600 > TIME_LIMIT_HOURS                                   # e.g. "/kaggle/input/histmatx-part1/histmatx_outputs" to reuse finished runs
CHI_LOCAL_CSV = ""   # not needed: the raw data come from the main experiment's output

CFG = dict(
    CITIES=["CHI", "NYC", "SF"], START_DATE="2019-01-01",
    TEST_WEEKS=26, REPORTING_LAG_WEEKS=2, VAL_WEEKS_FULL=4,
    BUDGETS=[4, 8, 16, 52, "all"], SEEDS=[0, 1, 2, 3, 4],
    MODELS=["Prior", "HistHeur", "LogReg", "GBDT", "GBDT-pooled", "MLP", "TabM-mini", "FTT", "FTT-pre",
            "TabICL", "HMX-scratch", "HMX-pre"],            # HMX-zero (zero-shot) is added automatically
    TEST_MAX_ROWS=120_000, VAL_MAX_ROWS=50_000, TRAIN_MAX_ROWS=500_000, PRETRAIN_MAX_ROWS=1_200_000,
    POOLED_SOURCE_ROWS=200_000, LOGREG_MAX_ROWS=200_000,
    TABICL_CONTEXT=10_000, TABICL_BUDGETS=[4, 8, 16, 52], TABICL_SEEDS=[0, 1, 2],
    PRETRAIN_EPOCHS=4, MAX_EPOCHS=60, MIN_EPOCHS=15, PATIENCE=5, STEP_TARGET=2_500_000,
    BIG_BUDGET_FTT_SEEDS=[0, 1, 2],   # FT-Transformer is ~4x slower than the others: 3 seeds at budgets 52/all
    D_MODEL=128, N_LAYERS=4, FTT_D=96, FTT_LAYERS=3,
    RUN_ABLATIONS=False, ABL_SEEDS=[0, 1, 2], ABL_BUDGETS=[8], ABL_PRETRAIN_ROWS=600_000,
    RUN_IMPORTANCE=True, N_BOOT=1000, CELL_KM=1.0, ALPHA=5.0, N_NEIGHBOURS=5,
)
if RUN_MODE == "full":     # window 2: short budgets only, 3 seeds, main models (MLP dropped: never the best target-only model at <= 16 weeks)
    CFG.update(BUDGETS=[4, 8, 16], SEEDS=[0, 1, 2], TABICL_SEEDS=[0, 1, 2], RUN_ABLATIONS=False, RUN_IMPORTANCE=False,
               MODELS=["Prior", "HistHeur", "LogReg", "GBDT", "GBDT-pooled", "TabM-mini", "FTT", "FTT-pre", "TabICL", "HMX-scratch", "HMX-pre"])
if RUN_MODE == "fast":
    CFG.update(SEEDS=[0, 1], BUDGETS=[4, 16, "all"], TEST_MAX_ROWS=60_000, TRAIN_MAX_ROWS=200_000,
               PRETRAIN_MAX_ROWS=500_000, TABICL_SEEDS=[0], TABICL_BUDGETS=[4, 16], ABL_SEEDS=[0],
               ABL_BUDGETS=[8], ABL_PRETRAIN_ROWS=300_000, PRETRAIN_EPOCHS=3, N_BOOT=500)
if RUN_MODE == "synthetic":
    CFG.update(SEEDS=[0, 1], BUDGETS=[4, 16, "all"], TEST_MAX_ROWS=6_000, VAL_MAX_ROWS=3_000, TRAIN_MAX_ROWS=20_000,
               PRETRAIN_MAX_ROWS=30_000, POOLED_SOURCE_ROWS=8_000, LOGREG_MAX_ROWS=10_000, TABICL_SEEDS=[0],
               TABICL_BUDGETS=[4], TABICL_CONTEXT=1_000, PRETRAIN_EPOCHS=2, MAX_EPOCHS=6, PATIENCE=2,
               STEP_TARGET=60_000, MIN_EPOCHS=4, D_MODEL=32, N_LAYERS=2, FTT_D=24, FTT_LAYERS=1, ABL_SEEDS=[0],
               ABL_BUDGETS=[4], ABL_PRETRAIN_ROWS=20_000, N_BOOT=200, TEST_WEEKS=12)

ON_KAGGLE = Path("/kaggle/working").exists()
OUT = Path("/kaggle/working/histmatx_w2" if ON_KAGGLE else "histmatx_w2")
for sub in ["data", "tables", "figures", "runs", "models", "logs"]:
    (OUT / sub).mkdir(parents=True, exist_ok=True)
if RESUME_FROM and Path(RESUME_FROM).exists():
    for sub in ["runs", "models", "data"]:
        src = Path(RESUME_FROM) / sub
        if src.exists():
            for f in src.iterdir():
                if f.is_file() and not f.name.endswith(".part") and not (OUT / sub / f.name).exists(): shutil.copy(f, OUT / sub / f.name)
    for f in ["results_main.csv", "results_ablation.csv", "importance.csv"]:
        s, d = Path(RESUME_FROM) / "tables" / f, OUT / "tables" / f
        if s.exists():
            old = pd.read_csv(s, dtype={"budget": str})
            if d.exists(): old = pd.concat([old, pd.read_csv(d, dtype={"budget": str})]).drop_duplicates("key", keep="last")
            old.to_csv(d, index=False)
    print("resumed from", RESUME_FROM)
if ON_KAGGLE and RUN_MODE == "full":
    if not any((OUT / "data").glob("*.csv")):
        print("Raw data of the main experiment not found in the input: the data will be downloaded again from the portals "
              "(needs Internet On; this can take up to an hour). The test window is fixed to weeks 337-362, so it is identical either way.")
    else: print("raw data reused from the main experiment:", sorted(f.name for f in (OUT / "data").glob("*.csv")))
    if (OUT / "tables" / "results_main.csv").exists():
        _r = pd.read_csv(OUT / "tables" / "results_main.csv", dtype={"budget": str})
        print("window-2 results already available per city:", _r.groupby("target").size().to_dict())
    else: print("window 2: starting fresh (only raw data reused)")

def _purge_reruns():
    p = OUT / "tables" / "results_main.csv"
    if not p.exists() or not RERUN_RULES: return
    r = pd.read_csv(p, dtype={"budget": str})
    if "protocol" not in r.columns: r["protocol"] = 1
    bad = r.apply(lambda x: any(x.target == t and x.budget == b and x.model in ms and x.protocol < v for t, b, ms, v in RERUN_RULES), axis=1)
    for k in r.key[bad]:
        f = OUT / "runs" / (re.sub(r"[^A-Za-z0-9_.-]", "_", f"main__{k}") + ".npz")
        if f.exists(): f.unlink()
    r[~bad].to_csv(p, index=False)
    if bad.sum(): print(f"queued {int(bad.sum())} runs from an older protocol for recomputation:", sorted(set(zip(r.target[bad], r.budget[bad]))))
_purge_reruns()

import torch, torch.nn as nn, torch.nn.functional as F
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEV.type == "cuda"
if DEV.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0)); torch.backends.cuda.matmul.allow_tf32 = True
else:
    print("No GPU found; deep models will be slow." + ("" if RUN_MODE == "synthetic" else " Switch the accelerator to GPU T4 x2."))
try:
    import xgboost as xgb
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost"]); import xgboost as xgb
XGB_DEVICE = "cuda" if DEV.type == "cuda" else "cpu"
print(f"mode={RUN_MODE} targets={TARGETS} torch={torch.__version__} xgboost={xgb.__version__} out={OUT}")
json.dump({"RUN_MODE": RUN_MODE, "TARGETS": TARGETS, **{k: v for k, v in CFG.items()}}, open(OUT / "logs" / "config.json", "w"), indent=1, default=str)

def set_seed(s):
    import random; random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if DEV.type == "cuda": torch.cuda.manual_seed_all(s)

## 2. Data
Three Socrata portals, downloaded year by year with only the needed columns. Chicago: *Crimes – 2001 to Present* (`ijzp-q8t2`). New York: *NYPD Complaint Data Historic* (`qgea-i56i`) plus *Current Year To Date* (`5uac-w243`). San Francisco: *Police Department Incident Reports: 2018 to Present* (`wg3w-h783`), which has one row per incident code, so it is de-duplicated by `incident_id`. A finished download is reused when the notebook is re-run.

In [ ]:
SOURCES = {
    "CHI": [dict(api="https://data.cityofchicago.org/resource/ijzp-q8t2.csv", date="date",
                 cols="id,date,primary_type,location_description,community_area,latitude,longitude")],
    "NYC": [dict(api="https://data.cityofnewyork.us/resource/qgea-i56i.csv", date="cmplnt_fr_dt",
                 cols="cmplnt_num,cmplnt_fr_dt,cmplnt_fr_tm,ofns_desc,prem_typ_desc,addr_pct_cd,latitude,longitude"),
            dict(api="https://data.cityofnewyork.us/resource/5uac-w243.csv", date="cmplnt_fr_dt",
                 cols="cmplnt_num,cmplnt_fr_dt,cmplnt_fr_tm,ofns_desc,prem_typ_desc,addr_pct_cd,latitude,longitude")],
    "SF":  [dict(api="https://data.sfgov.org/resource/wg3w-h783.csv", date="incident_datetime",
                 cols="incident_id,incident_datetime,incident_category,analysis_neighborhood,latitude,longitude")],
}

def soda_download(src, dest, start_year, page=500_000):
    import urllib.request, urllib.parse
    tmp = Path(str(dest) + ".part"); total, wrote_header, t0 = 0, False, time.time()
    with open(tmp, "wb") as out:
        for yr in range(start_year, pd.Timestamp.today().year + 1):
            offset, size, n_year = 0, page, 0
            while True:
                q = {"$select": src["cols"], "$where": f"{src['date']} between '{yr}-01-01T00:00:00' and '{yr}-12-31T23:59:59'",
                     "$order": ":id", "$limit": size, "$offset": offset}
                url = src["api"] + "?" + urllib.parse.urlencode(q); data, last = None, None
                for attempt in range(5):
                    try:
                        with urllib.request.urlopen(url, timeout=900) as r: data = r.read()
                        break
                    except Exception as e:
                        last = e; time.sleep(15 * (attempt + 1))
                if data is None: raise RuntimeError(f"request failed for {yr}: {last!r}")
                head, _, body = data.partition(b"\n")
                n = body.count(b"\n") + (0 if (body.endswith(b"\n") or not body) else 1)
                if not wrote_header: out.write(head.strip() + b"\n"); wrote_header = True
                if body: out.write(body if body.endswith(b"\n") else body + b"\n")
                n_year += n; offset += n
                if n == 0: break
                if n < size:
                    if n in (1000, 50000): size = n; continue
                    break
            print(f"    {yr}: {n_year:,} rows ({time.time() - t0:.0f}s)"); total += n_year
    if total == 0: raise RuntimeError("no rows returned")
    tmp.replace(dest); return total

def make_synthetic(seed=7):
    """Synthetic incidents in each portal's raw schema. Cities share the underlying structure (so transfer can help)
    but differ in label mix and size. Used only to test the pipeline."""
    rng = np.random.default_rng(seed)
    days = pd.date_range(CFG["START_DATE"], "2026-09-30", freq="D")
    K = 10; Wz = rng.normal(0, 1, (3, K)); slot_eff = rng.normal(0, .8, (6, K)); prem_of_type = rng.dirichlet(np.ones(9) * .5, K)
    raw_type = {"CHI": [["BATTERY", "ASSAULT"], ["ROBBERY"], ["CRIMINAL SEXUAL ASSAULT"], ["THEFT"], ["BURGLARY"], ["MOTOR VEHICLE THEFT"],
                        ["CRIMINAL DAMAGE"], ["DECEPTIVE PRACTICE"], ["NARCOTICS"], ["WEAPONS VIOLATION"], ["OTHER OFFENSE"]],
                "NYC": [["FELONY ASSAULT", "ASSAULT 3 & RELATED OFFENSES"], ["ROBBERY"], ["SEX CRIMES"], ["PETIT LARCENY", "GRAND LARCENY"],
                        ["BURGLARY"], ["GRAND LARCENY OF MOTOR VEHICLE"], ["CRIMINAL MISCHIEF & RELATED OF"], ["OFFENSES INVOLVING FRAUD"],
                        ["DANGEROUS DRUGS"], ["DANGEROUS WEAPONS"], ["HARRASSMENT 2"]],
                "SF": [["Assault"], ["Robbery"], ["Sex Offense"], ["Larceny Theft"], ["Burglary"], ["Motor Vehicle Theft"],
                       ["Malicious Mischief"], ["Fraud"], ["Drug Offense"], ["Weapons Offense"], ["Non-Criminal"]]}
    raw_prem = {"CHI": ["STREET", "APARTMENT", "SMALL RETAIL STORE", "PARKING LOT / GARAGE (NON RESIDENTIAL)", "CTA TRAIN",
                        "SCHOOL - PUBLIC BUILDING", "PARK PROPERTY", "HOSPITAL BUILDING / GROUNDS", "VACANT LOT / LAND"],
                "NYC": ["STREET", "RESIDENCE - APT. HOUSE", "CHAIN STORE", "PARKING LOT/GARAGE (PUBLIC)", "TRANSIT - NYC SUBWAY",
                        "PUBLIC SCHOOL", "PARK/PLAYGROUND", "HOSPITAL", "OTHER"]}
    spec = {"CHI": (30, 41.85, -87.68, 150), "NYC": (30, 40.72, -73.95, 220), "SF": (20, 37.76, -122.44, 70)}
    city_bias = {"CHI": rng.normal(0, .5, K), "NYC": rng.normal(0, .5, K), "SF": rng.normal(0, .5, K)}
    for city, (A, lat0, lon0, rate) in spec.items():
        lat_a = lat0 + rng.normal(0, .05, A); lon_a = lon0 + rng.normal(0, .05, A); z = rng.normal(0, 1, (A, 3))
        n_day = rng.poisson(rate / 7 * (1 + .2 * np.sin(np.arange(len(days)) / 58)), len(days)); N = n_day.sum()
        d = np.repeat(days.values, n_day); area = rng.integers(0, A, N); slot = rng.choice(6, N, p=[.12, .08, .15, .2, .25, .2])
        drift = (np.arange(N) / N)[:, None] * rng.normal(0, .4, K)[None]
        logit = city_bias[city][None] + z[area] @ Wz + slot_eff[slot] + drift
        p = np.exp(logit - logit.max(1, keepdims=True)); p /= p.sum(1, keepdims=True)
        typ = (p.cumsum(1) > rng.random((N, 1))).argmax(1); typ = np.where(rng.random(N) < .08, 10, typ)
        prem = np.array([rng.choice(9, p=prem_of_type[min(t, 9)]) for t in typ])
        hour = slot * 4 + rng.integers(0, 4, N); minute = rng.integers(0, 60, N)
        ts = pd.to_datetime(d) + pd.to_timedelta(hour, "h") + pd.to_timedelta(minute, "m")
        ts = ts.where(rng.random(N) > .03, pd.to_datetime(d))  # some unknown times recorded as 00:00
        lat = lat_a[area] + rng.normal(0, .01, N); lon = lon_a[area] + rng.normal(0, .01, N)
        rt = np.array([rng.choice(raw_type[city][t]) for t in typ], dtype=object)
        iso = pd.Series(ts).dt.strftime("%Y-%m-%dT%H:%M:%S.000")
        if city == "CHI":
            df = pd.DataFrame(dict(id=np.arange(N), date=iso, primary_type=rt, location_description=np.array(raw_prem["CHI"])[prem],
                                   community_area=area + 1, latitude=lat, longitude=lon))
        elif city == "NYC":
            df = pd.DataFrame(dict(cmplnt_num=np.arange(N), cmplnt_fr_dt=pd.Series(ts).dt.strftime("%Y-%m-%dT00:00:00.000"),
                                   cmplnt_fr_tm=pd.Series(ts).dt.strftime("%H:%M:%S"), ofns_desc=rt,
                                   prem_typ_desc=np.array(raw_prem["NYC"])[prem], addr_pct_cd=area * 2 + 1, latitude=lat, longitude=lon))
        else:
            df = pd.DataFrame(dict(incident_id=np.arange(N), incident_datetime=iso, incident_category=rt,
                                   analysis_neighborhood=np.array([f"Hood {i}" for i in range(A)])[area], latitude=lat, longitude=lon))
            df = pd.concat([df, df.sample(frac=.1, random_state=1)])  # SF repeats incidents with several codes
        df.to_csv(OUT / "data" / f"{city}.csv", index=False)

def load_raw(city):
    dest = OUT / "data" / f"{city}.csv"
    if dest.exists(): return pd.read_csv(dest, low_memory=False)
    if RUN_MODE == "synthetic":
        make_synthetic(); return pd.read_csv(dest, low_memory=False)
    start_year = int(CFG["START_DATE"][:4]); parts = []
    try:
        for i, src in enumerate(SOURCES[city]):
            p = OUT / "data" / f"{city}_{i}.csv"
            if not p.exists():
                print(f"  downloading {city} part {i} from {src['api']}"); soda_download(src, p, start_year)
            parts.append(pd.read_csv(p, low_memory=False))
    except Exception as e:
        print(f"  DOWNLOAD FAILED for {city}: {e!r}"[:300])
        if city == "CHI" and Path(CHI_LOCAL_CSV).exists():
            print("  using the local Chicago CSV instead (check that its end date is recent enough)")
            df = pd.read_csv(CHI_LOCAL_CSV, low_memory=False, usecols=["ID", "Date", "Primary Type", "Location Description", "Community Area", "Latitude", "Longitude"])
            df.columns = ["id", "date", "primary_type", "location_description", "community_area", "latitude", "longitude"]
            df.to_csv(dest, index=False); return df
        raise RuntimeError(f"No data for {city}. Turn Internet on (Settings → Internet → On) and re-run.")
    df = pd.concat(parts, ignore_index=True); df.to_csv(dest, index=False); return df

RAW = {}
for c in CFG["CITIES"]:
    t0 = time.time(); RAW[c] = load_raw(c); print(f"{c}: {len(RAW[c]):,} raw rows ({time.time() - t0:.0f}s)")

## 3. Harmonisation: one taxonomy, one premise scheme, one time grid
Crime labels differ by city ("BATTERY" vs "ASSAULT 3", "LARCENY THEFT" vs "PETIT LARCENY"). Ordered keyword rules map every city to **10 types in 3 groups**; offences outside them (homicide, harassment, "other offense", non-criminal) are left out, which keeps the label space identical across cities. Premises map to 9 groups plus *unknown*; San Francisco records no premise, so it is always *unknown*. Times exactly at 00:00:00 are treated as *unknown time* (portals use midnight as a placeholder). The full raw → harmonised mapping with counts is exported for auditing.

In [ ]:
TYPES = ["ASSAULT", "ROBBERY", "SEX_OFFENSE", "THEFT", "BURGLARY", "VEHICLE_THEFT", "DAMAGE", "FRAUD", "DRUGS", "WEAPONS"]
GROUP_NAMES = ["VIOLENT", "PROPERTY", "OTHER"]
TYPE_GROUP = np.array([0, 0, 0, 1, 1, 1, 1, 2, 2, 2])
K, S = len(TYPES), 6
SLOT_NAMES = ["00-04", "04-08", "08-12", "12-16", "16-20", "20-24"]
PREMS = ["STREET", "RESIDENCE", "COMMERCIAL", "PARKING_VEHICLE", "TRANSIT", "SCHOOL", "PARK_PUBLIC", "INSTITUTION", "OTHER"]
NP_ = len(PREMS)                      # index NP_ = unknown premise
TYPE_RULES = [(r"MOTOR VEHICLE|VEHICLE THEFT|STOLEN VEHICLE", "VEHICLE_THEFT"), (r"ROBBERY", "ROBBERY"), (r"BURGLAR", "BURGLARY"),
              (r"SEX|RAPE", "SEX_OFFENSE"), (r"WEAPON", "WEAPONS"), (r"NARCOTIC|DRUG", "DRUGS"),
              (r"FRAUD|DECEPTIVE|FORGERY|EMBEZZLE|COUNTERFEIT", "FRAUD"), (r"CRIMINAL DAMAGE|MISCHIEF|VANDAL", "DAMAGE"),
              (r"ASSAULT|BATTERY", "ASSAULT"), (r"THEFT|LARCENY", "THEFT")]
PREM_RULES = [(r"POLICE|FIRE STATION|HOSPITAL|MEDICAL|CLINIC|GOVERNMENT|JAIL|CHURCH|WORSHIP|LIBRARY|FEDERAL|COURT", "INSTITUTION"),
              (r"GAS STATION", "COMMERCIAL"),
              (r"\bCTA\b|TRANSIT|SUBWAY|\bBUS\b|TRAIN|STATION|AIRPORT|PLATFORM|RAILROAD|TAXI|FERRY", "TRANSIT"),
              (r"^RESIDEN|APARTMENT|APT|HOUSE|DWELLING|PORCH|YARD|HALLWAY|BASEMENT|NURSING|SHELTER|PUBLIC HOUSING|\bCHA\b|HOMELESS", "RESIDENCE"),
              (r"PARKING|GARAGE|VEHICLE|DRIVEWAY|AUTO", "PARKING_VEHICLE"),
              (r"SCHOOL|COLLEGE|UNIVERSITY|DAY CARE", "SCHOOL"),
              (r"STORE|SHOP|RESTAURANT|\bBAR\b|TAVERN|BANK|CURRENCY|OFFICE|COMMERCIAL|RETAIL|SUPERMARKET|GROCERY|DEPARTMENT|PHARMACY|HOTEL|MOTEL|CLUB|SALON|BEAUTY|LAUNDRY|CLEANER|THEATER|THEATRE|MALL|BUSINESS|FACTORY|WAREHOUSE|CONSTRUCTION|DELI|BODEGA|FAST FOOD|LIQUOR|PAWN|CHECK CASH|\bATM\b|LOAN|MARKET|BOUTIQUE|GYM", "COMMERCIAL"),
              (r"PARK|PLAYGROUND|BEACH|LAKE|FOREST|POOL|ATHLETIC|SPORTS", "PARK_PUBLIC"),
              (r"STREET|SIDEWALK|ALLEY|HIGHWAY|EXPRESSWAY|BRIDGE|OUTSIDE|STOOP|ROAD", "STREET")]

def _clean(series): return series.map(lambda x: "" if pd.isna(x) else str(x)).astype(object).str.upper().str.strip()
def map_values(series, rules, labels, default):
    u = pd.Series(_clean(series).unique())
    out = {}
    for v in u:
        lab = default
        if v not in ("NAN", "", "NONE", "(NULL)", "UNKNOWN", "N/A"):
            for pat, name in rules:
                if re.search(pat, v): lab = name; break
            else:
                lab = default if default is not None else None
        else:
            lab = None if default is None else "__UNK__"
        out[v] = lab
    return _clean(series).map(out), out

def parse_dt(s):
    try: return pd.to_datetime(s, format="ISO8601", errors="coerce")
    except Exception: return pd.to_datetime(s, errors="coerce")

ORIGIN = pd.Timestamp("2018-12-31")   # a Monday; week 0
AUDIT, MAPTAB = {}, []
def harmonise(city, raw):
    r = raw.copy(); n0 = len(r)
    if city == "CHI":
        if "id" in r: r = r.drop_duplicates("id")
        dt = parse_dt(r["date"])
        if dt.isna().mean() > .5: dt = pd.to_datetime(r["date"], format="%m/%d/%Y %I:%M:%S %p", errors="coerce")
        known = ~((dt.dt.hour == 0) & (dt.dt.minute == 0) & (dt.dt.second == 0))
        typ_raw, prem_raw, area_raw = r["primary_type"], r["location_description"], pd.to_numeric(r["community_area"], errors="coerce")
        area_raw = area_raw.where(area_raw > 0)
    elif city == "NYC":
        if "cmplnt_num" in r: r = r.drop_duplicates("cmplnt_num", keep="last")
        dt = pd.to_datetime(r["cmplnt_fr_dt"].astype(str).str[:10], format="%Y-%m-%d", errors="coerce")
        tm = r["cmplnt_fr_tm"].astype(str)
        hh = pd.to_numeric(tm.str[:2], errors="coerce"); mm = pd.to_numeric(tm.str[3:5], errors="coerce")
        dt = dt + pd.to_timedelta(hh.fillna(0), "h") + pd.to_timedelta(mm.fillna(0), "m")
        known = hh.notna() & ~(tm.str[:8] == "00:00:00")
        typ_raw, prem_raw, area_raw = r["ofns_desc"], r["prem_typ_desc"], pd.to_numeric(r["addr_pct_cd"], errors="coerce")
    else:
        if "incident_id" in r: r = r.drop_duplicates("incident_id")
        dt = parse_dt(r["incident_datetime"])
        known = ~((dt.dt.hour == 0) & (dt.dt.minute == 0))
        typ_raw, prem_raw = r["incident_category"], pd.Series(np.nan, index=r.index)
        area_raw = r["analysis_neighborhood"].where(r["analysis_neighborhood"].astype(str).str.upper().isin(["NAN", "NONE", ""]) == False)
    typ, tmap = map_values(typ_raw, TYPE_RULES, TYPES, None)
    prem, pmap = map_values(prem_raw, PREM_RULES, PREMS, "OTHER")
    for v, lab in tmap.items(): MAPTAB.append(dict(city=city, field="type", raw=v, mapped=lab))
    for v, lab in pmap.items(): MAPTAB.append(dict(city=city, field="premise", raw=v, mapped=lab))
    lat = pd.to_numeric(r["latitude"], errors="coerce"); lon = pd.to_numeric(r["longitude"], errors="coerce")
    d = pd.DataFrame(dict(dt=dt.values, known=known.values, type=typ.values, prem=prem.values, area_raw=area_raw.values,
                          lat=lat.values, lon=lon.values))
    d = d[d.dt.notna() & (d.dt >= pd.Timestamp(CFG["START_DATE"])) & d.type.notna() & pd.notna(d.area_raw)].copy()
    d["type"] = d.type.map({t: i for i, t in enumerate(TYPES)}).astype(np.int8)
    d["prem"] = d.prem.map({p: i for i, p in enumerate(PREMS)}).fillna(NP_).astype(np.int8)
    d["slot"] = np.where(d.known, d.dt.dt.hour // 4, S).astype(np.int8)
    d["wd"] = d.dt.dt.weekday.astype(np.int8); d["month"] = (d.dt.dt.month - 1).astype(np.int8)
    from pandas.tseries.holiday import USFederalHolidayCalendar
    hol = USFederalHolidayCalendar().holidays(d.dt.min(), d.dt.max())
    d["hol"] = d.dt.dt.normalize().isin(hol).astype(np.int8)
    d["week"] = ((d.dt - ORIGIN).dt.days // 7).astype(np.int32)
    codes, uniq = pd.factorize(d.area_raw.astype(str), sort=True); d["area"] = codes.astype(np.int32)
    ok = d.lat.between(d.lat.quantile(.001) - .05, d.lat.quantile(.999) + .05) & d.lon.between(d.lon.quantile(.001) - .05, d.lon.quantile(.999) + .05) & (d.lat.abs() > 1)
    lat0, lon0, coslat = d.lat[ok].min(), d.lon[ok].min(), math.cos(math.radians(d.lat[ok].mean()))
    iy = np.floor((d.lat - lat0) * 111.0 / CFG["CELL_KM"]); ix = np.floor((d.lon - lon0) * 111.0 * coslat / CFG["CELL_KM"])
    ck = np.where(ok, iy.fillna(0).astype(np.int64) * 10_000 + ix.fillna(0).astype(np.int64), -1)
    cc, cu = pd.factorize(pd.Series(ck).where(ck >= 0)); d["cell"] = cc.astype(np.int32)
    d["has_xy"] = ok.values
    AUDIT[city] = dict(raw_rows=n0, kept_rows=len(d), first=str(d.dt.min().date()), last=str(d.dt.max().date()),
                       n_areas=len(uniq), n_cells=len(cu), unknown_time=float((d.slot == S).mean()),
                       unknown_premise=float((d.prem == NP_).mean()), missing_xy=float((~ok).mean()))
    meta = dict(n_area=len(uniq), n_cell=len(cu), area_names=list(map(str, uniq)))
    # area centroids, k nearest neighbours, majority area of each cell
    A, C = meta["n_area"], meta["n_cell"]; okv = ok.values
    w = np.bincount(d.area.values[okv], minlength=A).astype(float) + 1e-9
    clat = np.bincount(d.area.values[okv], d.lat.values[okv], A) / w; clon = np.bincount(d.area.values[okv], d.lon.values[okv], A) / w
    D = np.hypot((clat[:, None] - clat[None]) * 111, (clon[:, None] - clon[None]) * 111 * coslat); np.fill_diagonal(D, np.inf)
    meta["nbrs"] = np.argsort(D, 1)[:, :min(CFG["N_NEIGHBOURS"], A - 1)]
    mc = d.cell.values >= 0
    ca = np.bincount(d.cell.values[mc].astype(np.int64) * A + d.area.values[mc], minlength=C * A).reshape(C, A)
    meta["cell_area"] = ca.argmax(1)
    return d[["week", "wd", "month", "hol", "slot", "prem", "type", "area", "cell", "dt"]].reset_index(drop=True), meta

DATA, META = {}, {}
for c in CFG["CITIES"]:
    DATA[c], META[c] = harmonise(c, RAW[c]); print(c, AUDIT[c])
del RAW; gc.collect()
pd.DataFrame(MAPTAB).to_csv(OUT / "tables" / "label_mapping_audit.csv", index=False)

# common time grid: the last complete week shared by all cities, minus a reporting-lag buffer
def last_full_week(d):
    mx = d.dt.max().normalize(); return int(((mx - ORIGIN).days + 1) // 7 - 1)
LAST_WEEK = min(last_full_week(DATA[c]) for c in CFG["CITIES"]) - CFG["REPORTING_LAG_WEEKS"] - WINDOW_SHIFT   # window 2: 26 weeks earlier
if EXPECTED_TEST_WINDOW:      # fixed window (also when the data had to be re-downloaded and now reach further)
    assert LAST_WEEK >= EXPECTED_TEST_WINDOW[1], f"data end too early for window 2 (last usable week {LAST_WEEK})"
    LAST_WEEK = EXPECTED_TEST_WINDOW[1]
T0 = LAST_WEEK - CFG["TEST_WEEKS"] + 1
NW = LAST_WEEK + 1
for c in CFG["CITIES"]:
    DATA[c] = DATA[c][DATA[c].week <= LAST_WEEK].reset_index(drop=True)
    META[c]["start_week"] = int(DATA[c].week.min())
wk = lambda w: (ORIGIN + pd.Timedelta(weeks=int(w))).date()
_tw = OUT / "logs" / "test_window.json"; _tw_src = Path(RESUME_FROM) / "logs" / "test_window.json" if RESUME_FROM else None
if _tw_src is not None and _tw_src.exists() and not _tw.exists(): shutil.copy(_tw_src, _tw)
if not _tw.exists() and EXPECTED_TEST_WINDOW:
    json.dump(dict(T0=EXPECTED_TEST_WINDOW[0], LAST_WEEK=EXPECTED_TEST_WINDOW[1]), open(_tw, "w"))
if _tw.exists():
    _old = json.load(open(_tw))
    if (_old["T0"], _old["LAST_WEEK"]) != (T0, LAST_WEEK):
        raise RuntimeError(f"Test window changed: earlier sessions used weeks {_old['T0']}-{_old['LAST_WEEK']}, this session computed {T0}-{LAST_WEEK}. "
                           "This happens when the data were re-downloaded. Add the FULL output of the previous session (not the zip) as input "
                           "and set RESUME_FROM to its histmatx_outputs folder, so the same data files are reused.")
else:
    json.dump(dict(T0=int(T0), LAST_WEEK=int(LAST_WEEK), start=str(wk(T0)), end=str(wk(LAST_WEEK + 1))), open(_tw, "w"))
print(f"test window: weeks {T0}–{LAST_WEEK}  ({wk(T0)} to {wk(LAST_WEEK + 1) - pd.Timedelta(days=1)})")
CITY_ID = {c: i for i, c in enumerate(CFG["CITIES"])}
AREA_OFF, off = {}, 0
for c in CFG["CITIES"]: AREA_OFF[c] = off; off += META[c]["n_area"]
N_AREA_TOT = off

### 3.1 Data audit and label shift between cities
The type mix differs strongly between cities (label shift). This is what makes naive transfer hard and why the evaluation always temperature-scales on target validation data.

In [ ]:
import matplotlib; matplotlib.use("Agg") if not hasattr(sys, "ps1") and "ipykernel" not in sys.modules else None
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 9, "axes.titlesize": 10, "legend.fontsize": 8})
PAL = {"CHI": "#2a6f97", "NYC": "#c44536", "SF": "#e09f3e"}

audit = pd.DataFrame(AUDIT).T; audit.index.name = "city"
audit["test_rows"] = [int((DATA[c].week >= T0).sum()) for c in audit.index]
audit["rows_per_week"] = [int(len(DATA[c]) / (NW - META[c]["start_week"])) for c in audit.index]
audit.to_csv(OUT / "tables" / "data_audit.csv")
try: display(audit)
except NameError: print(audit)

mix = pd.DataFrame({c: np.bincount(DATA[c].type, minlength=K) / len(DATA[c]) for c in CFG["CITIES"]}, index=TYPES)
smix = pd.DataFrame({c: np.bincount(DATA[c].slot[DATA[c].slot < S], minlength=S) / (DATA[c].slot < S).sum() for c in CFG["CITIES"]}, index=SLOT_NAMES)
mix.to_csv(OUT / "tables" / "type_mix_by_city.csv"); smix.to_csv(OUT / "tables" / "slot_mix_by_city.csv")
RARE = {c: list(np.argsort(np.bincount(DATA[c].type[DATA[c].week < T0], minlength=K))[:3]) for c in CFG["CITIES"]}

fig, ax = plt.subplots(1, 3, figsize=(13, 3.4), gridspec_kw=dict(width_ratios=[2.2, 1.2, 2]))
mix.plot.bar(ax=ax[0], color=[PAL[c] for c in mix.columns], width=.8); ax[0].set_ylabel("share of incidents"); ax[0].set_title("(a) Crime-type mix")
smix.plot.bar(ax=ax[1], color=[PAL[c] for c in smix.columns], width=.8, legend=False); ax[1].set_title("(b) Time-of-day mix")
for c in CFG["CITIES"]:
    s = DATA[c].groupby("week").size(); ax[2].plot(s.index, s.values, color=PAL[c], lw=.8, label=c)
ax[2].axvspan(T0, LAST_WEEK, color="grey", alpha=.15, label="test window")
for b in [4, 16, 52]: ax[2].axvline(T0 - b, color="grey", lw=.6, ls=":")
ax[2].set_yscale("log"); ax[2].set_title("(c) Weekly incidents; dotted = start of 4/16/52-week histories"); ax[2].legend(ncol=2)
plt.tight_layout(); plt.savefig(OUT / "figures" / "fig01_data_overview.png", bbox_inches="tight"); plt.show()

## 4. Online, city-agnostic history features
For an incident in area *a*, 1-km cell *c* and week *w*, every feature uses only incidents from weeks **before** *w* (lag ≥ 1 week), and only from weeks on or after the start of the history the target city is allowed to have. So a city with an 8-week history really has at most 8 weeks of past data, and during the test period its history grows week by week, as it would in deployment.

| Token group | dim | Content |
|---|---|---|
| `a4t`, `a52t` | 10, 10 | area type mix over the last 4 / 52 weeks (smoothed towards the city mix, log scale) |
| `a4s`, `a52s` | 6, 6 | area time-of-day mix over the last 4 / 52 weeks |
| `c52t` | 10 | 1-km cell type mix over 52 weeks (smoothed towards its area) |
| `n4t` | 10 | type mix of the 5 nearest areas over 4 weeks |
| `ci4t`, `ci4s` | 10, 6 | whole-city type and time mix over 4 weeks |
| `vol` | 5 | volumes relative to the city average (area 4 wk, area 52 wk, cell, neighbours) and the fraction of a year of history available |

All of these mean the same thing in every city, which is what lets a model trained in Chicago and New York be applied in San Francisco. Area identity is a separate, city-specific token.

In [ ]:
NUM_SPEC = [("a4t", K), ("a52t", K), ("a4s", S), ("a52s", S), ("c52t", K), ("n4t", K), ("ci4t", K), ("ci4s", S), ("vol", 5)]
GROUP_SL, _o = {}, 0
for g, dim in NUM_SPEC: GROUP_SL[g] = (_o, _o + dim); _o += dim
N_NUM = _o; G = len(NUM_SPEC)
CAT_FIELDS = ["wd", "month", "hol", "slot", "prem", "type", "area", "city"]
CAT_VOCAB = [7, 12, 2, S + 1, NP_ + 1, K, N_AREA_TOT, len(CFG["CITIES"])]
FPOS = {f: i for i, f in enumerate(CAT_FIELDS)}

def _win(cs, L):
    idx = np.arange(cs.shape[1] - 1); lo = np.maximum(idx - L, 0)
    return cs[:, idx] - cs[:, lo]
def _cum(X):
    cs = np.zeros((X.shape[0], X.shape[1] + 1, X.shape[2]), np.float32); np.cumsum(X, axis=1, out=cs[:, 1:]); return cs
def _share(x, prior, alpha): return (x + alpha * prior) / (x.sum(-1, keepdims=True) + alpha)

class CityTables:
    def __init__(self, city, hist_start):
        d, m = DATA[city], META[city]; A, C, W, al = m["n_area"], max(m["n_cell"], 1), NW, CFG["ALPHA"]
        sel = d.week.values >= hist_start
        a = d.area.values[sel].astype(np.int64); w = d.week.values[sel].astype(np.int64)
        t = d.type.values[sel].astype(np.int64); s = d.slot.values[sel].astype(np.int64); c = d.cell.values[sel].astype(np.int64)
        AT = np.bincount((a * W + w) * K + t, minlength=A * W * K).reshape(A, W, K).astype(np.float32)
        ms = s < S; AS = np.bincount((a[ms] * W + w[ms]) * S + s[ms], minlength=A * W * S).reshape(A, W, S).astype(np.float32)
        mc = c >= 0; CT = np.bincount((c[mc] * W + w[mc]) * K + t[mc], minlength=C * W * K).reshape(C, W, K).astype(np.float32)
        csA, csS, csC = _cum(AT), _cum(AS), _cum(CT); del AT, AS, CT
        a4t, a52t, a4s, a52s, c52t = _win(csA, 4), _win(csA, 52), _win(csS, 4), _win(csS, 52), _win(csC, 52)
        ci4t, ci52t, ci4s, ci52s = a4t.sum(0), a52t.sum(0), a4s.sum(0), a52s.sum(0)
        n4t = a4t[m["nbrs"]].sum(1)
        p4, p52, ps4, ps52 = _share(ci4t, 1 / K, K), _share(ci52t, 1 / K, K), _share(ci4s, 1 / S, S), _share(ci52s, 1 / S, S)
        a52sh = _share(a52t, p52[None], al)
        self.T = dict(a4t=np.log(_share(a4t, p4[None], al)), a52t=np.log(a52sh), a4s=np.log(_share(a4s, ps4[None], al)),
                      a52s=np.log(_share(a52s, ps52[None], al)), c52t=np.log(_share(c52t, a52sh[m["cell_area"]], al)),
                      n4t=np.log(_share(n4t, p4[None], al)))
        self.city = dict(ci4t=np.log(p4), ci4s=np.log(ps4))
        tot = lambda x: x.sum(-1); kn = m["nbrs"].shape[1]
        self.vol = dict(a4=np.log((tot(a4t) + 1) / (tot(ci4t) / A + 1)[None]), a52=np.log((tot(a52t) + 1) / (tot(ci52t) / A + 1)[None]),
                        c52=np.log((tot(c52t) + 1) / (tot(ci52t) / C + 1)[None]), n4=np.log((tot(n4t) + 1) / (kn * tot(ci4t) / A + 1)[None]))
        self.hist_start = hist_start
    def gather(self, area, week, cell):
        hc = cell >= 0; cc = np.where(hc, cell, 0)
        c52 = np.where(hc[:, None], self.T["c52t"][cc, week], self.T["a52t"][area, week])
        vc = np.where(hc, self.vol["c52"][cc, week], self.vol["a52"][area, week])
        vol = np.stack([self.vol["a4"][area, week], self.vol["a52"][area, week], vc, self.vol["n4"][area, week],
                        np.clip((week - self.hist_start) / 52, 0, 1)], 1)
        parts = [self.T["a4t"][area, week], self.T["a52t"][area, week], self.T["a4s"][area, week], self.T["a52s"][area, week],
                 c52, self.T["n4t"][area, week], self.city["ci4t"][week], self.city["ci4s"][week], vol]
        return np.concatenate(parts, 1).astype(np.float32)

_TAB_CACHE = {}
def tables(city, hist_start):
    key = (city, int(hist_start))
    if key not in _TAB_CACHE:
        if len(_TAB_CACHE) > 8: _TAB_CACHE.pop(next(iter(_TAB_CACHE)))
        _TAB_CACHE[key] = CityTables(city, hist_start)
    return _TAB_CACHE[key]

def make_block(city, rows, hist_start):
    d = DATA[city].iloc[rows]; tb = tables(city, hist_start)
    area, week, cell = d.area.values.astype(np.int64), d.week.values.astype(np.int64), d.cell.values.astype(np.int64)
    cat = np.stack([d.wd.values, d.month.values, d.hol.values, d.slot.values, d.prem.values, d.type.values,
                    area + AREA_OFF[city], np.full(len(d), CITY_ID[city])], 1).astype(np.int64)
    return dict(cat=cat, num=tb.gather(area, week, cell), week=week, area=area, city=city, n=len(d))

def cat_blocks(bl):
    out = {k: np.concatenate([b[k] for b in bl]) for k in ["cat", "num", "week", "area"]}
    out.update(city="+".join(b["city"] for b in bl), n=sum(b["n"] for b in bl)); return out

def sample_rows(mask, cap, seed):
    idx = np.flatnonzero(mask)
    if len(idx) > cap: idx = np.sort(np.random.default_rng(seed).choice(idx, cap, replace=False))
    return idx

def stable_seed(*parts): return int(hashlib.md5("|".join(map(str, parts)).encode()).hexdigest()[:8], 16)

# quick leakage check: features of week w must not change when week w's own incidents are removed
_d = DATA[CFG["CITIES"][0]]; _r = np.flatnonzero(_d.week.values == T0)[:200]
_f1 = make_block(CFG["CITIES"][0], _r, META[CFG["CITIES"][0]]["start_week"])["num"]
_bak = DATA[CFG["CITIES"][0]]; DATA[CFG["CITIES"][0]] = _d[_d.week != T0].reset_index(drop=True); _TAB_CACHE.clear()
_f2 = CityTables(CFG["CITIES"][0], META[CFG["CITIES"][0]]["start_week"]).gather(_d.area.values[_r], _d.week.values[_r], _d.cell.values[_r])
DATA[CFG["CITIES"][0]] = _bak; _TAB_CACHE.clear()
assert np.allclose(_f1, _f2, atol=1e-5), "leakage: a feature depends on its own week"
print("leakage check passed: history features only use earlier weeks;", N_NUM, "numeric features in", G, "token groups")

## 5. Splits and budgets
* **Test:** the same 26 weeks in every city (the last complete weeks shared by all three portals, minus a 2-week reporting-lag buffer), up to `TEST_MAX_ROWS` incidents sampled once per city.
* **Target budget *B* weeks:** the target city is only allowed data from weeks `T0 − B … T0 − 1`. Its last ⌈B/5⌉ weeks (1–4) are validation, the rest training. `all` = every week since 2019 (last 4 weeks validation). Budget 0 = zero-shot.
* **Sources:** the other two cities, full history up to `T0` (last 4 weeks for validation). Source data never overlaps the target test window.

In [ ]:
def val_weeks(b): return 4 if b == "all" else max(1, min(4, round(b / 5)))
def budget_weeks(city, b): return (T0 - META[city]["start_week"]) if b == "all" else b
def hist_start_for(city, b): return META[city]["start_week"] if b == "all" else T0 - b
def target_rows(city, b):
    wkv = DATA[city].week.values; hs = hist_start_for(city, b); vw = val_weeks(b)
    tr = sample_rows((wkv >= hs) & (wkv < T0 - vw), CFG["TRAIN_MAX_ROWS"], stable_seed("tr", city, b))
    va = sample_rows((wkv >= T0 - vw) & (wkv < T0), CFG["VAL_MAX_ROWS"], stable_seed("va", city, b))
    return tr, va
TEST_ROWS = {c: sample_rows(DATA[c].week.values >= T0, CFG["TEST_MAX_ROWS"], stable_seed("te", c)) for c in CFG["CITIES"]}
split_tab = []
for c in CFG["CITIES"]:
    for b in CFG["BUDGETS"]:
        tr, va = target_rows(c, b); split_tab.append(dict(city=c, budget=b, train_weeks=budget_weeks(c, b) - val_weeks(b), val_weeks=val_weeks(b),
                                                          train_rows=len(tr), val_rows=len(va), test_rows=len(TEST_ROWS[c])))
split_tab = pd.DataFrame(split_tab); split_tab.to_csv(OUT / "tables" / "splits.csv", index=False); print(split_tab.to_string(index=False))

## 6. Models
All neural models see the same inputs: 6 categorical fields (weekday, month, holiday, the other target, premise, area) and the 73 history features. For O1 the time slot is an input and for O2 the crime type is an input, as in the thesis.

| Model | Type | Uses source cities? |
|---|---|---|
| Prior | target class frequencies | no |
| HistHeur | average of the area's 4-week and the cell's 52-week mix (O2: area 4/52-week slot mix) | no |
| LogReg | multinomial logistic regression | no |
| GBDT | XGBoost, early stopping | no |
| GBDT-pooled | XGBoost on source + target rows, target up-weighted | **yes** |
| MLP | embeddings + 3×256 MLP | no |
| TabM-mini | parameter-efficient ensemble of 32 MLPs (Gorishniy et al., ICLR'25), re-implemented | no |
| FTT | FT-Transformer (Gorishniy et al., NeurIPS'21), one token per feature | no |
| FTT-pre | FT-Transformer pretrained on the sources for the same task, then fine-tuned | **yes** |
| TabICL | TabICLv2 foundation model, in-context learning on ≤ 10 k target rows | pretrained on synthetic tables |
| HMX-scratch | HiST-MAT-X trained on the target only | no |
| **HMX-pre** | HiST-MAT-X, masked multi-attribute pretraining on the sources, fine-tuned on the target | **yes** |
| HMX-zero | HMX-pre without any target training (area and city tokens masked) | **yes** |

**HiST-MAT-X.** 8 categorical tokens (each with its own [MASK]) and 9 history-group tokens (a small MLP per group) form a 17-token sequence for a pre-norm transformer (4 layers, *d* = 128). To predict an attribute, its token is replaced by [MASK] and its output feeds that attribute's head. Pretraining picks a random target per row (type 45 %, time 35 %, premise 20 %) and also drops other tokens at random (area and city 30 %, others 10 %), which teaches the model to work without area identity (needed for zero-shot) and with missing fields (San Francisco's premise). The crime-type loss adds the **coherent hierarchy loss**: label smoothing that moves ε = 0.05 of the target mass to the true type's siblings, plus λ = 0.3 × the cross-entropy of the *group* probability obtained by summing type probabilities (so group and type predictions can never disagree).

In [ ]:
TASKS = ["type", "slot"]
IN_FIELDS = {"type": [0, 1, 2, 3, 4, 6], "slot": [0, 1, 2, 5, 4, 6]}
NCLS = {"type": K, "slot": S}
def labels(block, task):
    y = block["cat"][:, FPOS[task]]; return y, (y < NCLS[task])

# severity matrices: type = distance in the 2-level tree (0, 1 same group, 2 other group); slot = circular slot distance
SEV = {"type": np.where(np.arange(K)[:, None] == np.arange(K)[None], 0, np.where(TYPE_GROUP[:, None] == TYPE_GROUP[None], 1, 2)).astype(float),
       "slot": np.minimum(np.abs(np.arange(S)[:, None] - np.arange(S)[None]), S - np.abs(np.arange(S)[:, None] - np.arange(S)[None])).astype(float)}

class Tens:
    """Block tensors on the device (cached)."""
    def __init__(self, b):
        self.cat = torch.as_tensor(b["cat"], device=DEV); self.num = torch.as_tensor(b["num"], device=DEV); self.n = b["n"]

def norm_stats(num):
    mu = num.mean(0); sd = num.std(0) + 1e-3; return mu.astype(np.float32), sd.astype(np.float32)

class MLPNet(nn.Module):
    def __init__(self, task, d_emb=16, width=256, depth=3, drop=0.1, k=0):
        super().__init__(); self.task, self.k = task, k; self.fields = IN_FIELDS[task]
        self.embs = nn.ModuleList([nn.Embedding(CAT_VOCAB[f] + 1, d_emb) for f in self.fields])
        D = d_emb * len(self.fields) + N_NUM
        self.register_buffer("mu", torch.zeros(N_NUM)); self.register_buffer("sd", torch.ones(N_NUM))
        if k: self.r = nn.Parameter(torch.randint(0, 2, (k, D)).float() * 2 - 1)   # TabM-mini adapter, random-sign init
        layers, d_in = [], D
        for _ in range(depth): layers += [nn.Linear(d_in, width), nn.ReLU(), nn.Dropout(drop)]; d_in = width
        self.mlp = nn.Sequential(*layers); C = NCLS[task]
        if k: self.hw = nn.Parameter(torch.randn(k, width, C) * width ** -.5); self.hb = nn.Parameter(torch.zeros(k, C))
        else: self.head = nn.Linear(width, C)
    def forward(self, cat, num):
        x = torch.cat([e(cat[:, f]) for e, f in zip(self.embs, self.fields)] + [(num - self.mu) / self.sd], 1)
        if self.k:
            h = self.mlp(x[:, None, :] * self.r[None]); return torch.einsum("bkw,kwc->bkc", h, self.hw) + self.hb
        return self.head(self.mlp(x))

class FTT(nn.Module):
    def __init__(self, task, d=96, L=3, heads=8, drop=0.1):
        super().__init__(); self.task = task; self.fields = IN_FIELDS[task]
        self.embs = nn.ModuleList([nn.Embedding(CAT_VOCAB[f] + 1, d) for f in self.fields])
        self.nw = nn.Parameter(torch.randn(N_NUM, d) * .02); self.nb = nn.Parameter(torch.zeros(N_NUM, d))
        self.cls = nn.Parameter(torch.randn(1, 1, d) * .02)
        layer = nn.TransformerEncoderLayer(d, heads, 2 * d, drop, activation="gelu", batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, L, enable_nested_tensor=False)
        self.head = nn.Sequential(nn.LayerNorm(d), nn.ReLU(), nn.Linear(d, NCLS[task]))
        self.register_buffer("mu", torch.zeros(N_NUM)); self.register_buffer("sd", torch.ones(N_NUM))
    def forward(self, cat, num):
        xn = (num - self.mu) / self.sd
        tok = torch.cat([self.cls.expand(len(cat), -1, -1), torch.stack([e(cat[:, f]) for e, f in zip(self.embs, self.fields)], 1),
                         xn[..., None] * self.nw + self.nb], 1)
        return self.head(self.enc(tok)[:, 0])

HMX_DEFAULT = dict(pre_task_p=(.45, .35, .20), ft_task_p=(.5, .4, .1), p_drop=.10, p_area_pre=.30, p_area_ft=.05, p_num=.05,
                   eps=.05, lam_g=.3, always_mask_fields=(), always_mask_groups=(), sources=None, mask_unknown_prem=False)
HEAD_FIELD = {"type": "type", "slot": "slot", "prem": "prem"}
GROUP_MASK = torch.full((3, K), -1e4)
for k_, g_ in enumerate(TYPE_GROUP): GROUP_MASK[g_, k_] = 0.0
SIB = np.zeros((K, K), np.float32)
for i in range(K):
    sib = [j for j in range(K) if TYPE_GROUP[j] == TYPE_GROUP[i] and j != i]; SIB[i, sib] = 1 / len(sib)

class HMX(nn.Module):
    def __init__(self, d=128, L=4, heads=8, drop=0.1):
        super().__init__()
        self.cat_emb = nn.ModuleList([nn.Embedding(v + 1, d) for v in CAT_VOCAB])            # index v = [MASK]
        self.register_buffer("mask_id", torch.tensor(CAT_VOCAB))
        self.num_proj = nn.ModuleList([nn.Sequential(nn.Linear(dim, d), nn.GELU(), nn.Linear(d, d)) for _, dim in NUM_SPEC])
        self.num_mask = nn.Parameter(torch.randn(G, d) * .02)
        self.pos = nn.Parameter(torch.randn(len(CAT_FIELDS) + G, d) * .02)
        layer = nn.TransformerEncoderLayer(d, heads, 2 * d, drop, activation="gelu", batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, L, enable_nested_tensor=False); self.norm = nn.LayerNorm(d)
        self.heads = nn.ModuleDict({"h_type": nn.Linear(d, K), "h_slot": nn.Linear(d, S), "h_prem": nn.Linear(d, NP_)})
        self.register_buffer("mu", torch.zeros(N_NUM)); self.register_buffer("sd", torch.ones(N_NUM))
    def encode(self, cat, num, cmask, gmask):
        c = torch.where(cmask, self.mask_id[None].expand_as(cat), cat)
        tok = torch.stack([e(c[:, i]) for i, e in enumerate(self.cat_emb)], 1)
        xn = (num - self.mu) / self.sd
        g = torch.stack([p(xn[:, s:e]) for p, (s, e) in zip(self.num_proj, GROUP_SL.values())], 1)
        g = torch.where(gmask[..., None], self.num_mask[None].expand_as(g), g)
        return self.norm(self.enc(torch.cat([tok, g], 1) + self.pos))
    def logits(self, h, task): return self.heads["h_" + task](h[:, FPOS[HEAD_FIELD[task]]])

def hmx_masks(cat, task_of_row, cfg, train, phase):
    B = len(cat); cm = torch.zeros(B, len(CAT_FIELDS), dtype=torch.bool, device=cat.device)
    gm = torch.zeros(B, G, dtype=torch.bool, device=cat.device)
    if train:
        p = torch.full((len(CAT_FIELDS),), cfg["p_drop"], device=cat.device)
        pa = cfg["p_area_pre"] if phase == "pre" else cfg["p_area_ft"]; p[FPOS["area"]] = pa; p[FPOS["city"]] = pa
        cm |= torch.rand(B, len(CAT_FIELDS), device=cat.device) < p
        gm |= torch.rand(B, G, device=cat.device) < cfg["p_num"]
    for f in cfg["always_mask_fields"]: cm[:, FPOS[f]] = True
    if cfg.get("mask_unknown_prem"): cm[:, FPOS["prem"]] |= cat[:, FPOS["prem"]] == NP_
    for gname in cfg["always_mask_groups"]: gm[:, [n for n, _ in NUM_SPEC].index(gname)] = True
    for ti, t in enumerate(["type", "slot", "prem"]):
        rows = task_of_row == ti
        if rows.any(): cm[rows, FPOS[HEAD_FIELD[t]]] = True
    return cm, gm

def type_loss(logits, y, cfg):
    logp = F.log_softmax(logits.float(), -1)
    q = F.one_hot(y, K).float() * (1 - cfg["eps"]) + cfg["eps"] * torch.as_tensor(SIB, device=y.device)[y]
    loss = -(q * logp).sum(-1)
    if cfg["lam_g"] > 0:
        glp = torch.logsumexp(logp[:, None, :] + GROUP_MASK.to(y.device)[None], -1)
        loss = loss - cfg["lam_g"] * glp.gather(1, torch.as_tensor(TYPE_GROUP, device=y.device)[y][:, None]).squeeze(1)
    return loss

def hmx_batch_loss(model, cat, num, cfg, phase):
    B = len(cat); probs = cfg["pre_task_p"] if phase == "pre" else cfg["ft_task_p"]
    u = torch.rand(B, device=cat.device); task = torch.zeros(B, dtype=torch.long, device=cat.device)
    slot_ok, prem_ok = cat[:, FPOS["slot"]] < S, cat[:, FPOS["prem"]] < NP_
    task[(u >= probs[0]) & (u < probs[0] + probs[1]) & slot_ok] = 1
    task[(u >= probs[0] + probs[1]) & prem_ok] = 2
    cm, gm = hmx_masks(cat, task, cfg, True, phase)
    h = model.encode(cat, num, cm, gm); total = torch.zeros((), device=cat.device)
    for ti, t in enumerate(["type", "slot", "prem"]):
        rows = task == ti
        if not rows.any(): continue
        lg = model.logits(h[rows], t); y = cat[rows, FPOS[HEAD_FIELD[t]]]
        total = total + (type_loss(lg, y, cfg).sum() if t == "type" else F.cross_entropy(lg.float(), y, reduction="sum"))
    return total / B

@torch.no_grad()
def hmx_predict(model, T, task, cfg, extra_fields=(), extra_groups=(), bs=8192):
    model.eval(); out = []
    c2 = dict(cfg); c2["always_mask_fields"] = tuple(cfg["always_mask_fields"]) + tuple(extra_fields)
    c2["always_mask_groups"] = tuple(cfg["always_mask_groups"]) + tuple(extra_groups)
    for i in range(0, T.n, bs):
        cat, num = T.cat[i:i + bs], T.num[i:i + bs]
        tr = torch.full((len(cat),), ["type", "slot", "prem"].index(task), device=DEV)
        cm, gm = hmx_masks(cat, tr, c2, False, "ft")
        with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
            out.append(F.softmax(model.logits(model.encode(cat, num, cm, gm), task).float(), -1).cpu())
    return torch.cat(out).numpy()

@torch.no_grad()
def net_predict(model, T, bs=8192):
    model.eval(); out = []
    for i in range(0, T.n, bs):
        with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
            lg = model(T.cat[i:i + bs], T.num[i:i + bs]).float()
        out.append((F.softmax(lg, -1).mean(1) if lg.dim() == 3 else F.softmax(lg, -1)).cpu())
    return torch.cat(out).numpy()

def mean_ll(P, y): return float(-np.log(np.clip(P[np.arange(len(y)), y], 1e-12, 1)).mean())

def fit_loop(model, n_train, batch_loss, val_score, lr, epochs, patience, bs, seed, wd=1e-5, tag=""):
    """AdamW, 3 % warm-up then cosine decay, early stopping on a validation score (lower is better)."""
    g = torch.Generator(device=DEV); g.manual_seed(seed)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=wd)
    spe = max(1, math.ceil(n_train / bs)); total = spe * epochs; warm = max(1, int(.03 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / warm) * (.1 + .9 * .5 * (1 + math.cos(math.pi * min(1, s / total)))))
    scaler = torch.amp.GradScaler("cuda", enabled=AMP); best, best_state, bad, t0 = float("inf"), None, 0, time.time()
    for ep in range(epochs):
        model.train(); perm = torch.randperm(n_train, device=DEV, generator=g)
        for i in range(spe):
            idx = perm[i * bs:(i + 1) * bs]
            if len(idx) < 2: continue
            with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
                loss = batch_loss(idx)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); sched.step()
        v = val_score()
        if v < best - 1e-5: best, bad = v, 0; best_state = {k: t.detach().clone() for k, t in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    if best_state is not None: model.load_state_dict(best_state)
    return dict(val=best, epochs=ep + 1, secs=time.time() - t0, hit_cap=bool(bad < patience))

def epochs_for(n): return int(np.clip(round(CFG["STEP_TARGET"] / max(n, 1)), CFG["MIN_EPOCHS"], CFG["MAX_EPOCHS"]))
def bs_for(n): return 256 if n < 20_000 else (512 if n < 100_000 else (1024 if n < 200_000 else 2048))

# ---------- supervised nets (MLP, TabM-mini, FTT): train / pretrain / fine-tune
def train_net(kind, task, trB, vaB, seed, init=None, lr=None, epochs=None):
    set_seed(seed)
    model = (FTT(task, d=CFG["FTT_D"], L=CFG["FTT_LAYERS"]) if kind == "FTT" else MLPNet(task, k=32 if kind == "TabM-mini" else 0)).to(DEV)
    if init is not None: model.load_state_dict(init)
    else:
        mu, sd = norm_stats(trB["num"]); model.mu.copy_(torch.as_tensor(mu)); model.sd.copy_(torch.as_tensor(sd))
    ytr, oktr = labels(trB, task); yva, okva = labels(vaB, task)
    Ttr = Tens({k: v[oktr] if k in ("cat", "num") else v for k, v in trB.items() if k in ("cat", "num")} | {"n": int(oktr.sum())})
    Tva = Tens({k: v[okva] if k in ("cat", "num") else v for k, v in vaB.items() if k in ("cat", "num")} | {"n": int(okva.sum())})
    yt = torch.as_tensor(ytr[oktr], device=DEV); yv = yva[okva]
    def bl(idx):
        lg = model(Ttr.cat[idx], Ttr.num[idx])
        if lg.dim() == 3: return F.cross_entropy(lg.reshape(-1, lg.shape[-1]).float(), yt[idx].repeat_interleave(lg.shape[1]))
        return F.cross_entropy(lg.float(), yt[idx])
    vs = lambda: mean_ll(net_predict(model, Tva), yv)
    n = Ttr.n; info = fit_loop(model, n, bl, vs, lr or (1e-3 if kind != "FTT" else 5e-4), epochs or epochs_for(n), CFG["PATIENCE"], bs_for(n), seed)
    return model, info

# ---------- HiST-MAT-X: pretrain / fine-tune
def train_hmx(trB, vaB, seed, cfg, phase, init=None, lr=None, epochs=None):
    set_seed(seed)
    model = HMX(d=CFG["D_MODEL"], L=CFG["N_LAYERS"]).to(DEV)
    if init is not None: model.load_state_dict(init)
    else:
        mu, sd = norm_stats(trB["num"]); model.mu.copy_(torch.as_tensor(mu)); model.sd.copy_(torch.as_tensor(sd))
    Ttr, Tva = Tens(trB), Tens(vaB)
    yv = {t: labels(vaB, t) for t in TASKS}
    bl = lambda idx: hmx_batch_loss(model, Ttr.cat[idx], Ttr.num[idx], cfg, phase)
    def vs():
        tot = 0.0
        for t in (TASKS if not (phase == "pre" and cfg["pre_task_p"][1] == 0) else ["type"]):
            y, ok = yv[t]
            if ok.sum() == 0: continue
            P = hmx_predict(model, Tva, t, cfg); tot += mean_ll(P[ok], y[ok])
        return tot
    n = Ttr.n
    info = fit_loop(model, n, bl, vs, lr or (1e-3 if init is None else 3e-4),
                    epochs or (CFG["PRETRAIN_EPOCHS"] if phase == "pre" else epochs_for(n)), CFG["PATIENCE"], bs_for(n), seed)
    return model, info

### 6.1 Metrics, calibration and stored statistics
Every probabilistic model is **temperature-scaled on the target validation weeks** (Guo et al., 2017) before testing, so log-loss differences are not just calibration differences. Raw log-loss is stored too. For each run the notebook stores per-test-week sufficient statistics (log-loss sums, confusion matrices, top-k hits, Brier, expected severity) and per-area log-loss, so that any week-block bootstrap or subgroup analysis can be done later without re-running models.

In [ ]:
from scipy.optimize import minimize_scalar
def fit_temperature(P, y):
    lp = np.log(np.clip(P, 1e-12, 1))
    def nll(lt):
        z = lp / math.exp(lt); z = z - z.max(1, keepdims=True); z = z - np.log(np.exp(z).sum(1, keepdims=True))
        return -z[np.arange(len(y)), y].mean()
    return float(math.exp(minimize_scalar(nll, bounds=(-3, 3), method="bounded").x))
def apply_temperature(P, T):
    z = np.log(np.clip(P, 1e-12, 1)) / T; z -= z.max(1, keepdims=True); e = np.exp(z); return (e / e.sum(1, keepdims=True)).astype(np.float32)

def run_stats(P, y, weeks, areas, task, n_area):
    Kc = NCLS[task]; n = len(y); py = np.clip(P[np.arange(n), y], 1e-12, 1); nll = -np.log(py)
    pred = P.argmax(1); rank = (P > py[:, None]).sum(1)
    onehot = np.zeros_like(P); onehot[np.arange(n), y] = 1; brier = ((P - onehot) ** 2).sum(1)
    esev = (P * SEV[task][y]).sum(1)
    wu, wi = np.unique(weeks, return_inverse=True)
    st = dict(weeks=wu, n_w=np.bincount(wi, minlength=len(wu)), nll_w=np.bincount(wi, nll, len(wu)),
              top1_w=np.bincount(wi, pred == y, len(wu)), top3_w=np.bincount(wi, rank < 3, len(wu)),
              brier_w=np.bincount(wi, brier, len(wu)), esev_w=np.bincount(wi, esev, len(wu)),
              cm_w=np.bincount((wi * Kc + y) * Kc + pred, minlength=len(wu) * Kc * Kc).reshape(len(wu), Kc, Kc),
              n_a=np.bincount(areas, minlength=n_area), nll_a=np.bincount(areas, nll, n_area),
              hit_a=np.bincount(areas, pred == y, n_area))
    conf = P.max(1); b = np.minimum((conf * 15).astype(int), 14)
    st.update(ece_n=np.bincount(b, minlength=15), ece_conf=np.bincount(b, conf, 15), ece_hit=np.bincount(b, pred == y, 15))
    return st

def f1_from_cm(cm):
    tp = np.diagonal(cm, axis1=-2, axis2=-1).astype(float); sup = cm.sum(-1); prd = cm.sum(-2)
    f1 = np.where(sup + prd > 0, 2 * tp / np.maximum(sup + prd, 1), np.nan)
    return np.where(sup > 0, f1, np.nan)
def summarize(st, task, rare):
    cm = st["cm_w"].sum(0); n = st["n_w"].sum(); f1 = f1_from_cm(cm); D = SEV[task]
    mis = cm * (1 - np.eye(len(cm)))
    out = dict(LL=st["nll_w"].sum() / n, macroF1=np.nanmean(f1), acc=st["top1_w"].sum() / n, top3=st["top3_w"].sum() / n,
               brier=st["brier_w"].sum() / n, exp_severity=st["esev_w"].sum() / n,
               mistake_severity=(mis * D).sum() / max(mis.sum(), 1),
               ECE=np.abs(st["ece_hit"] - st["ece_conf"]).sum() / n,
               rareF1=np.nanmean(f1[rare]) if task == "type" else np.nan)
    return out

RES_MAIN = OUT / "tables" / "results_main.csv"; RES_ABL = OUT / "tables" / "results_ablation.csv"
def done_keys(path):
    return set(pd.read_csv(path).key) if path.exists() else set()
DONE = {"main": done_keys(RES_MAIN), "abl": done_keys(RES_ABL)}
def rkey(task, tgt, budget, model, seed): return f"{task}|{tgt}|{budget}|{model}|{seed}"

def record(table, task, tgt, budget, model, seed, P_val, y_val, P_test, teB, extra=None, T=None):
    y, ok = labels(teB, task); P_test = P_test[ok]
    if T is None: T = fit_temperature(P_val, y_val) if P_val is not None and len(y_val) else 1.0
    Pc = apply_temperature(P_test, T)
    st = run_stats(Pc, y[ok], teB["week"][ok], teB["area"][ok], task, META[tgt]["n_area"])
    key = rkey(task, tgt, budget, model, seed)
    np.savez_compressed(OUT / "runs" / (re.sub(r"[^A-Za-z0-9_.-]", "_", f"{table}__{key}") + ".npz"), **st)
    extra = dict(extra or {})
    row = dict(key=key, task=task, target=tgt, budget=str(budget), weeks=0 if budget == 0 else budget_weeks(tgt, budget), model=model, seed=seed,
               T=T, LL_raw=mean_ll(P_test, y[ok]), **summarize(st, task, RARE[tgt]),
               epochs=extra.pop("epochs", np.nan), secs=extra.pop("secs", np.nan), n_train=extra.pop("n_train", np.nan),
               protocol=PROTOCOL_VERSION, extra=json.dumps(extra, default=str))
    path = RES_MAIN if table == "main" else RES_ABL
    out = pd.DataFrame([row])
    if path.exists():
        cols = list(pd.read_csv(path, nrows=0).columns)
        if set(out.columns) - set(cols):          # new column: rewrite the file with the union of columns
            old = pd.read_csv(path, dtype={"budget": str}); pd.concat([old, out], ignore_index=True).to_csv(path, index=False)
        else:
            out.reindex(columns=cols).to_csv(path, mode="a", header=False, index=False)
    else:
        out.to_csv(path, index=False)
    DONE[table].add(key)
    return row

def load_stats(table, task, tgt, budget, model, seed):
    p = OUT / "runs" / (re.sub(r"[^A-Za-z0-9_.-]", "_", f"{table}__{rkey(task, tgt, budget, model, seed)}") + ".npz")
    return dict(np.load(p)) if p.exists() else None

### 6.2 Baseline wrappers

In [ ]:
def xgb_frame(B, task):
    cols = {}
    for f in IN_FIELDS[task]:
        cols[CAT_FIELDS[f]] = pd.Categorical(B["cat"][:, f], categories=np.arange(CAT_VOCAB[f]))
    df = pd.DataFrame(cols); num = pd.DataFrame(B["num"], columns=[f"x{i}" for i in range(N_NUM)])
    return pd.concat([df, num], axis=1)

def run_xgb(task, trB, vaB, teB, seed, weight=None):
    ytr, ok = labels(trB, task); yva, okv = labels(vaB, task)
    dtr = xgb.DMatrix(xgb_frame({k: (v[ok] if isinstance(v, np.ndarray) and len(v) == trB["n"] else v) for k, v in trB.items()}, task),
                      label=ytr[ok], weight=None if weight is None else weight[ok], enable_categorical=True)
    dva = xgb.DMatrix(xgb_frame({k: (v[okv] if isinstance(v, np.ndarray) and len(v) == vaB["n"] else v) for k, v in vaB.items()}, task),
                      label=yva[okv], enable_categorical=True)
    params = dict(objective="multi:softprob", num_class=NCLS[task], eta=0.08, max_depth=7, min_child_weight=5, subsample=.8,
                  colsample_bytree=.8, reg_lambda=1.0, tree_method="hist", device=XGB_DEVICE, max_cat_to_onehot=1, max_bin=128,
                  eval_metric="mlogloss", seed=seed, nthread=-1, verbosity=0)
    bst = xgb.train(params, dtr, 1500 if RUN_MODE != "synthetic" else 200, evals=[(dva, "va")], early_stopping_rounds=50, verbose_eval=False)
    it = (0, bst.best_iteration + 1)
    pv = bst.predict(dva, iteration_range=it); pt = bst.predict(xgb.DMatrix(xgb_frame(teB, task), enable_categorical=True), iteration_range=it)
    return pv, yva[okv], pt, dict(n_trees=it[1])

def run_logreg(task, trB, vaB, teB, seed):
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    from scipy import sparse
    ytr, ok = labels(trB, task); yva, okv = labels(vaB, task)
    idx = np.flatnonzero(ok)
    if len(idx) > CFG["LOGREG_MAX_ROWS"]: idx = np.sort(np.random.default_rng(seed).choice(idx, CFG["LOGREG_MAX_ROWS"], replace=False))
    f = IN_FIELDS[task]; enc = OneHotEncoder(categories=[np.arange(CAT_VOCAB[i]) for i in f], handle_unknown="ignore")
    sc = StandardScaler()
    X = lambda B, r, fit=False: sparse.hstack([enc.fit_transform(B["cat"][r][:, f]) if fit else enc.transform(B["cat"][r][:, f]),
                                               sparse.csr_matrix(sc.fit_transform(B["num"][r]) if fit else sc.transform(B["num"][r]))]).tocsr()
    Xtr = X(trB, idx, True)
    clf = LogisticRegression(max_iter=300, C=1.0, random_state=seed).fit(Xtr, ytr[idx])
    def proba(B, r):
        p = np.full((len(r), NCLS[task]), 1e-6, np.float32); p[:, clf.classes_] = clf.predict_proba(X(B, r)); return p / p.sum(1, keepdims=True)
    return proba(vaB, np.flatnonzero(okv)), yva[okv], proba(teB, np.arange(teB["n"])), {}

def run_prior(task, trB, vaB, teB):
    ytr, ok = labels(trB, task); p = (np.bincount(ytr[ok], minlength=NCLS[task]) + 1.0); p = (p / p.sum()).astype(np.float32)
    yva, okv = labels(vaB, task)
    return np.tile(p, (okv.sum(), 1)), yva[okv], np.tile(p, (teB["n"], 1)), {}

def run_histheur(task, trB, vaB, teB):
    def pr(B):
        if task == "type":
            a, c = B["num"][:, slice(*GROUP_SL["a4t"])], B["num"][:, slice(*GROUP_SL["c52t"])]
        else:
            a, c = B["num"][:, slice(*GROUP_SL["a4s"])], B["num"][:, slice(*GROUP_SL["a52s"])]
        p = .5 * np.exp(a) + .5 * np.exp(c); return (p / p.sum(1, keepdims=True)).astype(np.float32)
    yva, okv = labels(vaB, task)
    return pr(vaB)[okv], yva[okv], pr(teB), {}

_TABICL_OK, _TABICL_FAILS = None, 0
def run_tabicl(task, trB, vaB, teB, seed):
    global _TABICL_OK, _TABICL_FAILS
    if _TABICL_FAILS >= 2:
        return None
    if _TABICL_OK is None:
        try:
            import tabicl  # noqa
            _TABICL_OK = True
        except ImportError:
            r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tabicl"], capture_output=True)
            try:
                import tabicl  # noqa
                _TABICL_OK = True
            except ImportError:
                _TABICL_OK = False; print("TabICL not available; skipped")
    if not _TABICL_OK: return None
    from tabicl import TabICLClassifier
    ytr, ok = labels(trB, task); yva, okv = labels(vaB, task)
    idx = np.flatnonzero(ok)
    if len(idx) > CFG["TABICL_CONTEXT"]: idx = np.sort(np.random.default_rng(seed).choice(idx, CFG["TABICL_CONTEXT"], replace=False))
    frame = lambda B, r: pd.concat([pd.DataFrame({CAT_FIELDS[f]: pd.Categorical(B["cat"][r, f].astype(str)) for f in IN_FIELDS[task]}),
                                    pd.DataFrame(B["num"][r], columns=[f"x{i}" for i in range(N_NUM)])], axis=1)
    try:
        clf = TabICLClassifier(random_state=seed, kv_cache=True, device=str(DEV) if DEV.type == "cuda" else "cpu")
        clf.fit(frame(trB, idx), ytr[idx])
    except Exception as e:
        _TABICL_FAILS += 1
        if _TABICL_FAILS >= 2: print("TabICL failed twice (checkpoint download needs Internet On); skipping it from now on")
        raise
    def proba(B, r):
        out = np.full((len(r), NCLS[task]), 1e-6, np.float32)
        for i in range(0, len(r), 20_000):
            rr = r[i:i + 20_000]; out[i:i + len(rr)][:, np.asarray(clf.classes_, int)] = clf.predict_proba(frame(B, rr))
        return out / out.sum(1, keepdims=True)
    vr = np.flatnonzero(okv)[:20_000]
    return proba(vaB, vr), yva[vr], proba(teB, np.arange(teB["n"])), dict(context=len(idx))

## 7. Pretraining on source cities (cached per target city and seed)

In [ ]:
def source_blocks(tgt, cap, sources=None, seed=0):
    srcs = sources or [c for c in CFG["CITIES"] if c != tgt]; tr, va = [], []
    for c in srcs:
        wkv = DATA[c].week.values; hs = META[c]["start_week"]
        r_tr = sample_rows(wkv < T0 - 4, cap // len(srcs), stable_seed("src", c, tgt, cap))
        r_va = sample_rows((wkv >= T0 - 4) & (wkv < T0), CFG["VAL_MAX_ROWS"] // len(srcs), stable_seed("srcva", c))
        tr.append(make_block(c, r_tr, hs)); va.append(make_block(c, r_va, hs))
    return cat_blocks(tr), cat_blocks(va)

_SRC_CACHE = {}
def get_sources(tgt, cap, sources=None):
    key = (tgt, cap, tuple(sources or ()))
    if key not in _SRC_CACHE:
        while len(_SRC_CACHE) >= 2: _SRC_CACHE.pop(next(iter(_SRC_CACHE)))
        _SRC_CACHE[key] = source_blocks(tgt, cap, sources)
    return _SRC_CACHE[key]

def pretrained(kind, tgt, seed, task=None, variant="full", overrides=None):
    """kind: 'HMX' (multi-attribute) or 'FTT' (supervised, one task). Returns (state_dict, info, source-val temperature)."""
    cfg = {**HMX_DEFAULT, **(overrides or {})}
    path = OUT / "models" / f"pre_{kind}_{task or 'multi'}_{variant}_{tgt}_s{seed}.pt"
    if path.exists():
        ck = torch.load(path, map_location="cpu", weights_only=False); return ck["state"], ck["info"], ck["T"]
    cap = CFG["PRETRAIN_MAX_ROWS"] if variant == "full" else CFG["ABL_PRETRAIN_ROWS"]
    trB, vaB = get_sources(tgt, cap, cfg["sources"])
    if kind == "HMX":
        model, info = train_hmx(trB, vaB, seed + 1000, cfg, "pre")
        Tva = Tens(vaB); T = {}
        for t in TASKS:
            y, ok = labels(vaB, t); T[t] = fit_temperature(hmx_predict(model, Tva, t, {**cfg, "always_mask_fields": ("area", "city")})[ok], y[ok])
    else:
        model, info = train_net("FTT", task, trB, vaB, seed + 1000, epochs=CFG["PRETRAIN_EPOCHS"]); T = None
    state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
    torch.save(dict(state=state, info=info, T=T), path)
    print(f"    pretrained {kind}/{task or 'multi'}/{variant} for target {tgt} seed {seed}: {info['epochs']} ep, val {info['val']:.4f}, {info['secs']:.0f}s")
    del model; return state, info, T

## 8. Main experiment
Loop: target city → budget → seed → model. Every finished run is appended to `tables/results_main.csv` and skipped on re-runs, so the notebook can be stopped and resumed (also across Kaggle sessions via `RESUME_FROM`).

In [ ]:
def run_zero_shot(tgt):
    """HMX-zero: the source-pretrained model applied with no target training. Area/city tokens and unrecorded premises are masked."""
    if "HMX-pre" not in CFG["MODELS"]: return
    teB = None
    for seed in CFG["SEEDS"]:
        if all(rkey(t, tgt, 0, "HMX-zero", seed) in DONE["main"] for t in TASKS): continue
        st, info, Tsrc = pretrained("HMX", tgt, seed)
        if teB is None: teB = make_block(tgt, TEST_ROWS[tgt], T0)
        model = HMX(d=CFG["D_MODEL"], L=CFG["N_LAYERS"]).to(DEV); model.load_state_dict(st)
        cfg0 = {**HMX_DEFAULT, "always_mask_fields": ("area", "city"), "mask_unknown_prem": True}
        for t in TASKS:
            record("main", t, tgt, 0, "HMX-zero", seed, None, None, hmx_predict(model, Tens(teB), t, cfg0), teB, T=Tsrc[t])
        del model

def run_target(tgt):
    teRows = TEST_ROWS[tgt]; t_start = time.time()
    run_zero_shot(tgt)
    for b in (RUN_BUDGETS or CFG["BUDGETS"]):
        hs = hist_start_for(tgt, b); trR, vaR = target_rows(tgt, b)
        trB, vaB, teB = make_block(tgt, trR, hs), make_block(tgt, vaR, hs), make_block(tgt, teRows, hs)
        print(f"[{tgt} | budget {b}] train {trB['n']:,}  val {vaB['n']:,}  test {teB['n']:,}  ({(time.time() - t_start) / 60:.1f} min)")
        for seed in CFG["SEEDS"]:
            for m in CFG["MODELS"]:
                if time_up(): print(f"time limit ({TIME_LIMIT_HOURS} h) reached: no new runs; re-run with RESUME_FROM to continue"); return
                tasks_todo = [t for t in TASKS if rkey(t, tgt, b, m, seed) not in DONE["main"]]
                if not tasks_todo: continue
                if m == "TabICL" and (b not in CFG["TABICL_BUDGETS"] or seed not in CFG["TABICL_SEEDS"]): continue
                if m in ("Prior", "HistHeur") and seed != CFG["SEEDS"][0]: continue     # deterministic
                if m in ("FTT", "FTT-pre") and str(b) in ("52", "all") and seed not in CFG["BIG_BUDGET_FTT_SEEDS"]: continue
                t0 = time.time()
                try:
                    if m.startswith("HMX"):
                        cfg = dict(HMX_DEFAULT)
                        if m == "HMX-pre":
                            st, _, _ = pretrained("HMX", tgt, seed); model, info = train_hmx(trB, vaB, seed, cfg, "ft", init=st)
                        else:
                            model, info = train_hmx(trB, vaB, seed, cfg, "ft")
                        Tva, Tte = Tens(vaB), Tens(teB)
                        for t in tasks_todo:
                            y, ok = labels(vaB, t)
                            record("main", t, tgt, b, m, seed, hmx_predict(model, Tva, t, cfg)[ok], y[ok], hmx_predict(model, Tte, t, cfg), teB,
                                   dict(epochs=info["epochs"], secs=time.time() - t0, n_train=trB["n"], hit_cap=info["hit_cap"]))
                        del model
                        continue
                    for t in tasks_todo:
                        t1 = time.time()
                        if m == "Prior": out = run_prior(t, trB, vaB, teB)
                        elif m == "HistHeur": out = run_histheur(t, trB, vaB, teB)
                        elif m == "LogReg": out = run_logreg(t, trB, vaB, teB, seed)
                        elif m == "GBDT": out = run_xgb(t, trB, vaB, teB, seed)
                        elif m == "GBDT-pooled":
                            sB, _ = get_sources(tgt, CFG["POOLED_SOURCE_ROWS"])
                            pB = cat_blocks([sB, trB]); w = np.concatenate([np.ones(sB["n"]), np.full(trB["n"], max(1.0, .5 * sB["n"] / max(trB["n"], 1)))])
                            out = run_xgb(t, pB, vaB, teB, seed, weight=w.astype(np.float32))
                        elif m in ("MLP", "TabM-mini", "FTT"):
                            model, info = train_net(m, t, trB, vaB, seed); y, ok = labels(vaB, t)
                            out = (net_predict(model, Tens({"cat": vaB["cat"][ok], "num": vaB["num"][ok], "n": int(ok.sum())})), y[ok],
                                   net_predict(model, Tens(teB)), dict(epochs=info["epochs"], hit_cap=info["hit_cap"])); del model
                        elif m == "FTT-pre":
                            st, _, _ = pretrained("FTT", tgt, seed, task=t)
                            model, info = train_net("FTT", t, trB, vaB, seed, init=st, lr=3e-4); y, ok = labels(vaB, t)
                            out = (net_predict(model, Tens({"cat": vaB["cat"][ok], "num": vaB["num"][ok], "n": int(ok.sum())})), y[ok],
                                   net_predict(model, Tens(teB)), dict(epochs=info["epochs"], hit_cap=info["hit_cap"])); del model
                        elif m == "TabICL":
                            out = run_tabicl(t, trB, vaB, teB, seed)
                            if out is None: continue
                        else: raise ValueError(m)
                        pv, yv, pt, extra = out
                        record("main", t, tgt, b, m, seed, pv, yv, pt, teB, dict(extra, secs=time.time() - t1, n_train=trB["n"]))
                except Exception as e:
                    print(f"   !! {m} failed on {tgt}/{b}/seed {seed}: {e!r}"[:400])
                    with open(OUT / "logs" / "errors.log", "a") as fh: fh.write(f"{tgt}|{b}|{m}|{seed}|{e!r}\n")
                if DEV.type == "cuda": torch.cuda.empty_cache()
        r = pd.read_csv(RES_MAIN, dtype={"budget": str}); r = r[(r.target == tgt) & (r.budget == str(b)) & (r.task == "type")]
        if len(r): print(r.groupby("model")[["LL", "macroF1"]].mean().sort_values("LL").round(4).T.to_string())

for tgt in CFG["CITIES"]:      # zero-shot for every city (cheap once the pretrained models exist)
    if tgt in TARGETS or (OUT / "models" / f"pre_HMX_multi_full_{tgt}_s{CFG['SEEDS'][0]}.pt").exists(): run_zero_shot(tgt)
for tgt in (TARGETS if RUN_MAIN else []):
    print(f"\n========== target city {tgt} ==========")
    run_target(tgt); gc.collect()

## Export

In [ ]:
r = pd.read_csv(RES_MAIN, dtype={"budget": str}) if RES_MAIN.exists() else pd.DataFrame()
if len(r):
    print("runs per city and budget:"); print(r.groupby(["target", "budget"]).size().unstack(fill_value=0))
    print(r[r.task == "type"].groupby(["target", "budget", "model"]).LL.mean().unstack().round(4).to_string())
zp = OUT.parent / "histmatx_w2.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in OUT.rglob("*"):
        if f.is_file() and "data" not in f.parts and f.suffix != ".pt": z.write(f, f.relative_to(OUT.parent))
print("saved", zp, f"({zp.stat().st_size / 1e6:.1f} MB)")